### Faiss Vector Store

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델(캐싱)

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace = underlying_embeddings.model
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [33]:
vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [34]:
vectorstore

In [36]:
vectorstore = None

In [37]:
vectorstore

In [ ]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [40]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [15]:
vectorstore = None

In [16]:
vectorstore

In [ ]:
FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [ ]:
vectorstore

In [22]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

##### `similarity_search` 메서드

In [41]:

# results = vectorstore.similarity_search(query, k=3) # k는 유사도 검색에서 반환할 상위 문서 개수(top‑k)
results = vectorstore.similarity_search(query, k=5)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

[결과 1]
1.2 Validity of Collected Data
본 연구에서는 의료기기 임상시험에 특화된 Private
수집된 데이터셋은 의료기기 임상시험에 특화된
LLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화
Private LLM 구축을 위해 도메인 적합성과 다양성, 그리
데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적
고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총
용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로
111,954페이지로 구성된 데이터는 의료기기 임상시험의
구성된다. 각 단계는 의
---
[결과 2]
Accuracy analysis.
인 강의 자료 등
Time Automated processes reduce the time
Efficiency required for analysis.  프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR
Detailed LLM provides detailed insights to (Clinical Study Report) 템플릿 등
Insights support medical decision-making.
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
Scalability L
---
[결과 3]
적 접근법이 임상시험에서 어떻게 효과적으로 적용될 수
직접적이고 중요한 영향을 미친다[11].
있는지를 제시하고 있으며, 앞으로 더 많은 사례 연구를
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
통해 LLM 기반 AI의 활용 가능성을 탐구할 계획이다.[15]
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
Table 5. Key Benefits of LLM-Based Analysis in
과 같이 분류된다:
Medical Device Clinical Tr
---
[결과 4]
괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수
를 통해 해당 분야에서 최적의 성능을 달성하도록

---

### Faiss 검색 메서드 작동 방식

1. **쿼리 임베딩**: 사용자가 입력한 `query` 텍스트를 임베딩 모델을 사용하여 벡터로 변환합니다.
2. **유사성 검색**: 변환된 쿼리 벡터와 벡터스토어 내의 모든 문서 벡터 간의 **거리**(Distance) 또는 **유사도**(Similarity)를 계산합니다.
    * FAISS는 기본적으로 **L2(유클리드) 거리**를 사용하며, **점수가 0에 가까울수록** 더 유사한 문서임을 의미합니다.
3. **결과 반환**: 계산된 유사도 점수를 기준으로 상위 $k$개의 문서 덩어리(`Document`)를 정렬하여 반환합니다.

### 검색 메서드 상세 비교

| 특징 | `as_retriever` | `similarity_search` |
| :--- | :--- | :--- |
| **반환 객체** | **`Retriever` (Runnable)** | `List[Document]` (문서 리스트) |
| **핵심 역할** | 검색 **전략**을 정의한 '도구' 생성 | 검색 '행위'를 즉시 수행 |
| **활용 환경** | **LCEL 체인(Chain)** 구성 시 필수 | 즉각적인 결과 확인, 단독 로직 |
| **인터페이스** | `.invoke()`, `.batch()`, `.stream()` 지원 | 메서드 호출 즉시 실행 |
| **추가 설정** | MMR, Score Threshold 등 검색 기법 설정 가능 | 검색 개수($k$) 등 기본 파라미터 위주 |

### `as_retriever()`와 Runnable 객체의 이해

`as_retriever()`를 호출하면 단순히 검색 기능을 가진 객체가 아니라, LangChain의 표준 인터페이스인 **`Runnable`** 객체가 생성됩니다. 

1. **체인 구성의 부품 (`|` 연산자)**:
   - `Runnable`이기 때문에 `retriever | prompt | llm`과 같이 파이프라인(`|`) 연산자를 사용하여 다른 컴포넌트와 유연하게 연결할 수 있습니다.
2. **표준화된 실행 메서드**:
   - **`.invoke(query)`**: 단일 질의에 대한 문서를 검색합니다.
   - **`.batch([q1, q2])`**: 여러 쿼리를 병렬로 처리하여 결과를 반환합니다.
3. **검색 전략의 추상화**:
   - 단순히 가장 가까운 벡터만 찾는 것이 아니라, **MMR(Max Marginal Relevance)** 방식을 선택하여 결과의 다양성을 확보하거나, 특정 점수 이상의 문서만 가져오도록(`score_threshold`) 설정을 미리 박아둘(Encapsulation) 수 있습니다.

#### `as_retriever()` 메서드

In [9]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [10]:
retriever

VectorStoreRetriever(tags=['FAISS', 'CacheBackedEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000196872DEAB0>, search_kwargs={})

In [12]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"

In [14]:
results = retriever.invoke(query)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

[결과 1]
1.2 Validity of Collected Data
본 연구에서는 의료기기 임상시험에 특화된 Private
수집된 데이터셋은 의료기기 임상시험에 특화된
LLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화
Private LLM 구축을 위해 도메인 적합성과 다양성, 그리
데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적
고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총
용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로
111,954페이지로 구성된 데이터는 의료기기 임상시험의
구성된다. 각 단계는 의
---
[결과 2]
Accuracy analysis.
인 강의 자료 등
Time Automated processes reduce the time
Efficiency required for analysis.  프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR
Detailed LLM provides detailed insights to (Clinical Study Report) 템플릿 등
Insights support medical decision-making.
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
Scalability L
---
[결과 3]
적 접근법이 임상시험에서 어떻게 효과적으로 적용될 수
직접적이고 중요한 영향을 미친다[11].
있는지를 제시하고 있으며, 앞으로 더 많은 사례 연구를
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
통해 LLM 기반 AI의 활용 가능성을 탐구할 계획이다.[15]
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
Table 5. Key Benefits of LLM-Based Analysis in
과 같이 분류된다:
Medical Device Clinical Tr
---
[결과 4]
괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수
를 통해 해당 분야에서 최적의 성능을 달성하도록

---

### FAISS 구현 방식 비교: 생성자 vs 팩토리 메서드

| 구분 | 생성자 방식 (`FAISS()`) | 팩토리 메서드 방식 (`from_documents()`) |
| --- | --- | --- |
| **추상화 수준** | **저수준 (Low-level)** | **고수준 (High-level)** |
| **제어 권한** | 인덱스 종류, 거리 측정 방식 직접 지정 가능 | 기본 설정( 거리, IndexFlat) 사용 |
| **초기 데이터** | 데이터 없이 **빈 상태**로 시작 가능 | 초기 생성 시 **데이터 리스트** 필수 |
| **구현 난이도** | 차원 계산, 도큐먼트 스토어 결합 등 직접 수행 | 내부 로직에 의해 모든 과정 자동화 |
| **주요 용도** | 대규모 최적화, 커스텀 인덱스(HNSW 등) 필요 시 | 빠른 프로토타이핑, 표준적인 RAG 구현 |

### 핵심 차이점

#### 1. 인덱스 최적화 및 제어 (Control)

* **생성자 방식:** `faiss.IndexFlatL2`, `IndexIVFFlat`, `IndexHNSWFlat` 등 FAISS가 제공하는 다양한 인덱스 알고리즘을 개발자가 직접 선택할 수 있습니다. 이는 데이터 규모나 검색 속도 요구사항에 따른 정밀한 튜닝을 가능하게 합니다.
* **`from_documents`:** 가장 보편적인 `IndexFlatL2`(전수 조사 방식)로 자동 고정됩니다. 소규모 데이터에는 적합하나, 수백만 건 이상의 대규모 데이터 처리 시 성능 최적화에 제약이 있습니다.

#### 2. 초기화 프로세스의 자동화 (Automation)

* **생성자 방식:** 임베딩 모델의 차원(-dimension)을 미리 계산하고, `InMemoryDocstore`와 ID 매핑 테이블을 수동으로 결합해야 하는 번거로움이 있습니다.
* **`from_documents`:** 입력된 문서를 분석하여 임베딩 차원을 자동으로 파악하고, 내부적으로 인덱스와 저장소를 즉시 생성합니다. 코드가 간결하며 실수할 확률이 적습니다.

#### 3. 운영 유연성 (Flexibility)

* **생성자 방식:** 빈 인덱스를 먼저 생성해 둔 뒤, 애플리케이션 실행 중에 동적으로 데이터를 추가(`add_documents`)하는 구조에 유리합니다.
* **`from_documents`:** 인스턴스 생성과 데이터 적재가 동시에 일어나므로, 이미 준비된 정적 데이터를 한꺼번에 벡터화할 때 효율적입니다.

### 무엇을 선택해야 하는가?

* **엔지니어링 측면의 최적화가 중요하거나, 실시간으로 문서를 추가해야 하는 환경**이라면 **생성자 방식**을 사용하여 인덱스 구조를 직접 설계하는 것이 옳습니다.
* **빠른 기능 구현이 우선이며, 일반적인 문서 검색 성능으로도 충분한 상황**이라면 **`from_documents`** 메서드를 사용하여 코드 복잡도를 낮추는 것을 권장합니다.

> https://docs.langchain.com/oss/python/integrations/vectorstores#faiss

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. 벡터 차원 정의: 사용 중인 임베딩 모델이 생성하는 결과물의 길이(차원)를 계산합니다.
# 임베딩 모델(embeddings)은 사전에 정의되어 있어야 합니다.
embedding_dim = len(embeddings.embed_query("hello world"))

# 2. FAISS 인덱스 초기화: 벡터 간의 거리 계산 방식을 결정합니다.
# IndexFlatL2: 유클리드 거리($L2$ distance)를 계산하는 가장 정확한 완전 탐색(Brute-force) 방식입니다.
index = faiss.IndexFlatL2(embedding_dim)

# 3. LangChain FAISS 객체 조립: 검색 엔진(index)과 원본 데이터 저장소(docstore)를 결합합니다.
vector_store = FAISS(
    embedding_function=embeddings,  # 텍스트를 벡터로 변환할 함수
    index=index,                   # 유사도 검색을 수행할 FAISS 인덱스
    docstore=InMemoryDocstore(),   # 실제 텍스트 내용과 메타데이터를 담을 메모리 저장소
    index_to_docstore_id={},       # 인덱스 번호와 저장소 ID 간의 매핑 테이블(초기화 시 빈 값)
)

# 4. 데이터 준비: 검색 대상이 될 문서 객체들을 생성합니다.
documents = [
    Document(page_content="컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.", metadata={"source": "edu"}),
    Document(page_content="인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.", metadata={"source": "tech"}),
    Document(page_content="고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.", metadata={"source": "pets"}),
    Document(page_content="파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.", metadata={"source": "tech"}),
]

# 5. 데이터 적재: 문서를 임베딩하여 벡터화한 뒤 FAISS 인덱스에 추가합니다.
# 내부적으로 임베딩 생성 -> 인덱스 저장 -> docstore 저장이 동시에 수행됩니다.
vector_store.add_documents(documents=documents)

# 6. 유사도 검색(Top-K): 질문과 의미적으로 가장 가까운 문서 k개를 추출합니다.
query = "머신러닝과 AI 기술에 대해 알려줘"
results = vector_store.similarity_search(query, k=2)

print(f"--- [검색 질의]: {query} ---")
for i, doc in enumerate(results):
    print(f"결과 {i+1}: {doc.page_content} (출처: {doc.metadata['source']})")

# 7. 점수 포함 검색: 거리 값(Distance)을 포함하여 검색 결과의 신뢰도를 확인합니다.
# IndexFlatL2를 사용하므로 점수(Score)는 거리를 의미하며, 0에 가까울수록 유사도가 높습니다.
results_with_score = vector_store.similarity_search_with_score(query, k=4)

print("\n--- [점수 포함 검색 결과] ---")
for doc, score in results_with_score:
    print(f"거리(Score): {score:.4f} | 내용: {doc.page_content}")

--- [검색 질의]: 머신러닝과 AI 기술에 대해 알려줘 ---
결과 1: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다. (출처: tech)
결과 2: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다. (출처: tech)

--- [점수 포함 검색 결과] ---
거리(Score): 0.5265 | 내용: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.
거리(Score): 0.6992 | 내용: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.
거리(Score): 0.7824 | 내용: 컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.
거리(Score): 0.8486 | 내용: 고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.


---